In [8]:
import os

import gymnasium as gym
import structlog
import torch
from samsara_rl.control.dyna.deep_dyna_q import (
    DeepDynaQ,
)
from samsara_rl.functions.neural_networks.fully_connected import FullyConnected
from samsara_rl.functions.neural_networks.world_model_interpreter import WorldModelInterpreter
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole
import torch
log = structlog.get_logger()

%load_ext tensorboard

In [9]:
class CartPoleInterpreter(WorldModelInterpreter):
        def __init__(self):
            self.state_len = 4

        def next_state_logits(self, output):
            return output[:, 0 : self.state_len]

        def predicted_next_state(self, output):
            return output[:, 0 : self.state_len]

        def predicted_reward(self, output):
            return output[:, self.state_len]

        def predicted_terminal(self, output):
            return output[:, self.state_len + 1]

        def parse(self, output):
            return (
                self.next_state_logits(output),
                self.predicted_reward(output),
                self.predicted_terminal(output),
            )

cart_pole_wm_interpreter = CartPoleInterpreter()

In [10]:
log = structlog.get_logger()

MAX_EPISODES = 2800
EVAL_EPISODES = 20
BASE_DIR = "logs/deep_dyna_q"
ALPHAS = [0.0001, 0.0003, 0.001]  # 0.005, 0.001, 0.0005, 0.0001, 0.00005]
GAMMAS = [0.99]

In [11]:
def run_experiment(env, alpha: float, gamma: float) -> None:
    """Train and evaluate a single actor-critic configuration.

    Creates value and policy networks with separate optimizers,
    trains for MAX_EPISODES episodes, and logs metrics to TensorBoard.

    Args:
        env: Gymnasium CartPole environment.
        alpha: Learning rate for the policy optimizer. Critic uses alpha * 1.75.
        gamma: Discount factor.
    """
    model_name = "deep_dyna_q"
    run_name = f"{model_name}_alpha={alpha}_gamma={gamma}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)
    cart_pole_wm = FullyConnected(6, 32, 6)
    q = FullyConnected(4,32,2)


    agent = DeepDynaQ(
        mdp=env,
        world_model=cart_pole_wm,
        world_model_interpreter=cart_pole_wm_interpreter,
        model_plan_steps=10,
        count_planning_updates=True,
        model_train_steps=70,
        warmup=80,
        gamma=gamma,
        q=q,
        alpha=alpha,
        state_loss=torch.nn.MSELoss(),
        target_update_freq=1000,
         log_dir=run_dir,
        experiment_name=run_name,
    )
    # agent.register(CartPoleLogger())
    agent.evaluate(max_iter=MAX_EPISODES)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [12]:
def main() -> None:
    """Run grid search over feature configs, bias, and learning rates.

    Iterates over all combinations of FEATURE_CONFIGS, BIAS_CONFIGS,
    and ALPHAS, training and evaluating each one.
    """
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for alpha in ALPHAS:
        for gamma in GAMMAS:
            run_experiment(env, alpha, gamma)

In [ ]:
main()

2026-10-24 16:18:03 [info     ] starting_run                   run='deep_dyna_q_alpha=0.0001_gamma=0.99'


/home/ashish/reinforcement-learning-library/samsara-rl/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


2026-10-24 16:18:58 [info     ] world_model_loss               episode=20 loss=9.563640343919285
2026-10-24 16:18:58 [info     ] world_model_loss               episode=20 loss=1.9255495663557134
2026-10-24 16:18:59 [info     ] world_model_loss               episode=20 loss=1.5440502763494943
2026-10-24 16:18:59 [info     ] world_model_loss               episode=20 loss=2.338878956633767
2026-10-24 16:18:59 [info     ] world_model_loss               episode=20 loss=2.6313590050863302
2026-10-24 16:18:59 [info     ] world_model_loss               episode=20 loss=3.2911212950434514
2026-10-24 16:18:59 [info     ] world_model_loss               episode=20 loss=6.606082677442494
2026-10-24 16:18:59 [info     ] world_model_loss               episode=20 loss=3.3754069764382844
2026-10-24 16:18:59 [info     ] world_model_loss               episode=20 loss=4.2519715899073995
2026-10-24 16:19:00 [info     ] world_model_loss               episode=20 loss=6.952756187621532
2026-10-24 16:19:00 [inf

2026-10-24 16:19:07 [info     ] world_model_loss               episode=20 loss=1.0557087932679419
2026-10-24 16:19:07 [info     ] world_model_loss               episode=20 loss=2.3771166388333835
2026-10-24 16:19:07 [info     ] world_model_loss               episode=20 loss=0.7722751077128016
2026-10-24 16:19:07 [info     ] world_model_loss               episode=20 loss=3.266770933780917
2026-10-24 16:19:08 [info     ] world_model_loss               episode=20 loss=3.3477346109524078
2026-10-24 16:20:22 [info     ] world_model_loss               episode=40 loss=0.560442422778159
2026-10-24 16:20:23 [info     ] world_model_loss               episode=40 loss=4.0473975779623075
2026-10-24 16:20:23 [info     ] world_model_loss               episode=40 loss=0.10405724810315117
2026-10-24 16:20:23 [info     ] world_model_loss               episode=40 loss=0.39316253341729596
2026-10-24 16:20:23 [info     ] world_model_loss               episode=40 loss=3.4000312463614266
2026-10-24 16:20:23 

2026-10-24 16:22:47 [info     ] world_model_loss               episode=80 loss=2.8755180662308826
2026-10-24 16:22:47 [info     ] world_model_loss               episode=80 loss=1.107059933060269
2026-10-24 16:22:47 [info     ] world_model_loss               episode=80 loss=0.31798799439776654
2026-10-24 16:22:47 [info     ] world_model_loss               episode=80 loss=1.5848085525704079
2026-10-24 16:22:48 [info     ] world_model_loss               episode=80 loss=1.5703913318275824
2026-10-24 16:22:48 [info     ] world_model_loss               episode=80 loss=1.764026941560375
2026-10-24 16:22:48 [info     ] world_model_loss               episode=80 loss=1.5003927155986205
2026-10-24 16:22:48 [info     ] world_model_loss               episode=80 loss=0.657171377875671
2026-10-24 16:22:48 [info     ] warmup_complete                episode=81
2026-10-24 16:22:48 [info     ] warmup_complete                episode=81
2026-10-24 16:22:49 [info     ] warmup_complete                episode

...

2026-10-24 21:04:10 [info     ] world_model_loss               episode=1320 loss=0.09333701219357393
2026-10-24 21:04:10 [info     ] world_model_loss               episode=1320 loss=0.28077983863097017
2026-10-24 21:04:11 [info     ] world_model_loss               episode=1320 loss=0.32272378775265304
2026-10-24 21:04:11 [info     ] world_model_loss               episode=1320 loss=2.95605065421496
2026-10-24 21:04:11 [info     ] world_model_loss               episode=1320 loss=1.0846980671286064
2026-10-24 21:04:11 [info     ] world_model_loss               episode=1320 loss=0.757630122078267
2026-10-24 21:04:11 [info     ] world_model_loss               episode=1320 loss=2.8549498184000366
2026-10-24 21:04:11 [info     ] world_model_loss               episode=1320 loss=0.022604992760404168
2026-10-24 21:04:12 [info     ] world_model_loss               episode=1320 loss=0.08251457582994064
2026-10-24 21:04:12 [info     ] world_model_loss               episode=1320 loss=0.81940872829601

2026-10-24 21:08:12 [info     ] world_model_loss               episode=1340 loss=0.04587663198259914
2026-10-24 21:08:12 [info     ] world_model_loss               episode=1340 loss=0.09240462584240934
2026-10-24 21:08:12 [info     ] world_model_loss               episode=1340 loss=0.047093234761498445
2026-10-24 21:08:12 [info     ] world_model_loss               episode=1340 loss=0.03796634774031094
2026-10-24 21:08:12 [info     ] world_model_loss               episode=1340 loss=0.13445513128095732
2026-10-24 21:08:13 [info     ] world_model_loss               episode=1340 loss=0.073580403832909
2026-10-24 21:08:13 [info     ] world_model_loss               episode=1340 loss=0.10591508680548549
2026-10-24 21:08:13 [info     ] world_model_loss               episode=1340 loss=3.188486884219728
2026-10-24 21:08:13 [info     ] world_model_loss               episode=1340 loss=0.16986771825009508
2026-10-24 21:08:13 [info     ] world_model_loss               episode=1340 loss=0.16226498036

2026-10-24 21:11:15 [debug    ] target_network_swap            episode=1357
2026-10-24 21:11:31 [debug    ] target_network_swap            episode=1359
2026-10-24 21:11:42 [info     ] world_model_loss               episode=1360 loss=0.08623209197847305
2026-10-24 21:11:42 [info     ] world_model_loss               episode=1360 loss=0.022253016651926194
2026-10-24 21:11:42 [info     ] world_model_loss               episode=1360 loss=0.04275600866039953
2026-10-24 21:11:42 [info     ] world_model_loss               episode=1360 loss=3.1567811854931453
2026-10-24 21:11:42 [info     ] world_model_loss               episode=1360 loss=0.955425110388028
2026-10-24 21:11:43 [info     ] world_model_loss               episode=1360 loss=0.0169190605263499
2026-10-24 21:11:43 [info     ] world_model_loss               episode=1360 loss=0.052489766773836134
2026-10-24 21:11:43 [info     ] world_model_loss               episode=1360 loss=3.0133596209684677
2026-10-24 21:11:43 [info     ] world_model

2026-10-24 21:11:50 [info     ] world_model_loss               episode=1360 loss=0.09306120812709846
2026-10-24 21:11:50 [info     ] world_model_loss               episode=1360 loss=0.09251964690972325
2026-10-24 21:11:50 [info     ] world_model_loss               episode=1360 loss=0.5885784393003829
2026-10-24 21:11:51 [info     ] world_model_loss               episode=1360 loss=0.9419425826190313
2026-10-24 21:11:51 [info     ] world_model_loss               episode=1360 loss=0.34791759295751506
2026-10-24 21:11:51 [info     ] world_model_loss               episode=1360 loss=2.9815942366072106
2026-10-24 21:11:51 [info     ] world_model_loss               episode=1360 loss=0.5871015381823735
2026-10-24 21:11:51 [info     ] world_model_loss               episode=1360 loss=0.0284178339051745
2026-10-24 21:11:51 [info     ] world_model_loss               episode=1360 loss=3.021906697590063
2026-10-24 21:11:52 [info     ] world_model_loss               episode=1360 loss=0.029484443717757

2026-10-24 21:11:59 [info     ] world_model_loss               episode=1360 loss=3.000456679119784
2026-10-24 21:11:59 [info     ] world_model_loss               episode=1360 loss=0.028105021606012404
2026-10-24 21:11:59 [info     ] world_model_loss               episode=1360 loss=0.06694110349122712
2026-10-24 21:11:59 [info     ] world_model_loss               episode=1360 loss=0.2075326017493923
2026-10-24 21:11:59 [info     ] world_model_loss               episode=1360 loss=0.034530835824590794
2026-10-24 21:12:00 [info     ] world_model_loss               episode=1360 loss=0.05949625112750942
2026-10-24 21:12:00 [info     ] world_model_loss               episode=1360 loss=3.042723702653281
2026-10-24 21:12:02 [debug    ] target_network_swap            episode=1361
2026-10-24 21:12:17 [debug    ] target_network_swap            episode=1363
2026-10-24 21:12:32 [debug    ] target_network_swap            episode=1364
2026-10-24 21:12:47 [debug    ] target_network_swap            episo